NOTE: Day 61 mein dekha tha ki model ko “exam” ek aise data par dena chahiye jo usne training mein kabhi dekha hi nahi. train_test_split data ko do hisso mein baant deta hai:

- Train set (80%): model isi se seekhta hai
- Test set (20%): sirf aakhri evaluation ke liye, tab tak ise haath nahi lagana

1. Data load karo aur X, y dobara banao (Day 62 ka output):

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

features = pd.read_csv('../data/processed/customer_ml_dataset.csv')

X = features[['Recency', 'Frequency', 'Monetary_log', 'AvgOrderValue', 'Tenure', 'UniqueProducts', 'Region']]
X = pd.get_dummies(X, columns=['Region'], drop_first=True)
y = features['Churned']

print(X.shape, y.shape)

(5283, 8) (5283,)


2. Basic split (pehle bina stratify ke, taaki fark dikhe):

In [2]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print("Overall churn %:", round(y.mean() * 100, 2))
print("Train churn %  :", round(y_tr.mean() * 100, 2))
print("Test churn %   :", round(y_te.mean() * 100, 2))

Overall churn %: 56.62
Train churn %  : 57.0
Test churn %   : 55.06


Dono percentages overall se thode alag aa sakte hain. Chhote ya imbalanced data mein yeh drift badhta hai.

3. Stratified split (yehi final version hai, Day 56B mein is par note chhoda tha):

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Overall churn %:", round(y.mean() * 100, 2))
print("Train churn %  :", round(y_train.mean() * 100, 2))
print("Test churn %   :", round(y_test.mean() * 100, 2))

Train shape: (4226, 8) | Test shape: (1057, 8)
Overall churn %: 56.62
Train churn %  : 56.63
Test churn %   : 56.58


note: stratify=y har split mein churned/not-churned ka same ratio rakhta hai. Ab test set ka score misleading nahi hoga.

4. Parameters ka matlab:

- test_size=0.2: 20% test. Dataset chhota ho to 0.25 bhi chalta hai
- random_state=42: seed fix karta hai, taaki har baar same split mile (reproducibility). Number kuch bhi ho sakta hai, bas same rakho
- stratify=y: class ratio preserve karta hai

5. Leakage ke 3 niyam jo aage kaam aayenge:

In [4]:
print("""
1. Split SABSE PEHLE karo, scaling/SMOTE/imputation se pehle.
2. StandardScaler sirf X_train par fit hoga (Day 64), phir X_test par sirf transform.
3. SMOTE sirf X_train par lagega (Day 64C). Test set ko kabhi resample nahi karte.
Test set ko "real duniya" ki tarah rehne do.
""")


1. Split SABSE PEHLE karo, scaling/SMOTE/imputation se pehle.
2. StandardScaler sirf X_train par fit hoga (Day 64), phir X_test par sirf transform.
3. SMOTE sirf X_train par lagega (Day 64C). Test set ko kabhi resample nahi karte.
Test set ko "real duniya" ki tarah rehne do.



Agar scaling pehle poore data par kar di, to test set ki information (mean/std) train mein leak ho jaati hai aur score jhootha ban jaata hai.

6. Row-level duplicate/overlap check:

In [5]:
print("Duplicate customers in features:", features['Customer ID'].duplicated().sum())   # 0 aana chahiye
print("Index overlap train/test:", len(set(X_train.index) & set(X_test.index)))        # 0 aana chahiye

Duplicate customers in features: 0
Index overlap train/test: 0


Humara data customer-level hai (ek row = ek customer), isliye ek hi customer train aur test dono mein nahi aa sakta.

7. Splits save karo (Day 64+ ke notebooks yahin se load karenge):

In [7]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

8. Naive baseline ek baar test set par dekh lo (Day 56B ka concept):

In [8]:
majority_share = y_test.value_counts(normalize=True).max()
print(f"Always-predict-majority accuracy on test: {majority_share:.3f}")

Always-predict-majority accuracy on test: 0.566


Yeh number note karo. Aage ka koi bhi model isse behtar hona chahiye, aur sirf accuracy se nahi, Recall/F1 se bhi.

practice questions:

1. random_state ko 42 se 7 karke dobara split karo. stratify=y ke saath train/test churn % kitna hilta hai?

Bilkul nahi hilta.
Agar aap stratify=y use kar rahe hain, toh chahe aap random_state ko 42 rakhein, 7 rakhein, ya koi bhi doosra number, train aur test set ka churn percentage bilkul same (ya exact overall percentage ke barabar) rahega.


Kyunki stratify=y ka kaam hi yhi hai ki woh random sampling ke bawajood class ratio (churned vs non-churned) ko force kare ki woh poore dataset ke ratio ke barabar hi maintain rahe. random_state sirf yeh decide karta hai ki kaun se specific customer rows train mein jayenge aur kaun se test mein, lekin proportions hamesha fixed rahenge.

2. Bina stratify ke 5 alag random_state values try karo aur test churn % ki range note karo. Stratify kitna fark daalta hai?

Agar aap stratify hata dete hain aur alag-alag seeds (random_state = 0, 42, 100, 500, 999) try karte hain, toh aap dekhenge ki test set ka churn % fluctuate (upar-neeche) karega.


- Range: Agar overall churn rate maan lijiye 68% hai, toh bina stratify ke alag-alag seeds par test set ka churn percentage shayad kahin 63% se lekar 73% tak vary kar sakta hai.

- Stratify kitna fark daalta hai? Chhote ya moderately imbalanced datasets mein yeh "data drift" model evaluation ko unreliable bana deta hai. Agar test set mein achanak churn rate kam ya zyada ho gaya, toh aapka model ka accuracy/F1-score sach mein model ki performance ki wajah se nahi, balki kharab split ki wajah se badal jayega. stratify=y is random variance ko khatam karke ek fair aur stable evaluation ensure karta hai.